## Ejercicio 1. Paradigma, tarea, algoritmo, predicción y decisión

**Objetivo:** distinguir los niveles que intervienen en un sistema de Machine Learning.

### Tabla de situaciones

| Situación | Paradigma | Tarea | Salida esperada | Algoritmos posibles | Decisión apoyada |
|---|---|---|---|---|---|
| 1. Estimar la especie de un pingüino a partir de sus medidas corporales | Supervisado | Clasificación multiclase | Etiqueta (Adelie, Chinstrap o Gentoo), o probabilidades por especie | Regresión logística, árbol de decisión, Random Forest, k-NN (vecino más cercano), SVC | Registrar la especie de un individuo, o derivar a revisión experta los casos de baja confianza |
| 2. Estimar la masa corporal de un pingüino | Supervisado | Regresión | Número continuo en gramos | Regresión lineal, Random Forest, k-NN (vecino más cercano), SVR, MLP | Estimar el estado físico o la alimentación necesaria sin pesar al animal |
| 3. Formar grupos de pingüinos con perfiles morfológicos semejantes | No supervisado | Clustering | Asignación a un grupo (sin nombre previo) | K-means, Agglomerative Clustering, Gaussianos, DBSCAN | Descubrir perfiles morfológicos para un estudio posterior |
| 4. Usar pocas especies verificadas y muchas observaciones sin especie confirmada | Semisupervisado | Clasificación con etiquetas parciales | Especie probable para las observaciones sin etiqueta | Entrenar un clasificador (Random Forest, k-NN, regresión logística) solo con las observaciones etiquetadas, o agrupar todo con Gaussianos o K-means y usar las etiquetadas para nombrar los grupos | Priorizar qué observaciones verificar en campo, ya que verificar es caro |

### Respuestas

**1. ¿Qué elementos adicionales al algoritmo definen el problema experimental?**
Definen el problema la unidad de análisis, la población objetivo, la variable objetivo, los predictores disponibles al momento de predecir, el escenario de uso, el criterio de partición, la métrica, el baseline y el costo de los errores. El mismo algoritmo con otra partición o métrica es otro experimento.

**2. ¿En qué se diferencian una predicción y una decisión?**
La predicción es la salida del modelo (por ejemplo, "70 % Gentoo"). La decisión es la acción que alguien toma con ella (por ejemplo, "lo registro como Gentoo" o "lo mando a revisar"). Un modelo puede predecir bien y aun así llevar a malas decisiones si ignora los costos de error o el umbral.

**3. ¿Puede un mismo algoritmo resolver más de una tarea?**
Sí. **Random Forest** clasifica especies (`RandomForestClassifier`) y predice masa corporal (`RandomForestRegressor`). Lo mismo ocurre con **k-NN** y con **MLP**. Cambia solo cómo combinan las respuestas: votan una clase o promedian un número.

**4. ¿Puede una misma tarea resolverse con distintos algoritmos?**
Sí. Para clasificar especies sirven la **regresión logística** (frontera lineal y probabilidades) o un **árbol de decisión** (reglas del tipo "si la aleta es mayor que X..."). Difieren en interpretabilidad y en si necesitan escalado: la logística sí, el árbol no.

**5. ¿Por qué la codificación de las especies como 0, 1 y 2 conserva la naturaleza clasificatoria de la tarea?**
Porque los números son solo etiquetas, no cantidades. Que Gentoo valga 2 y Adelie 0 no significa que Gentoo sea "el doble" ni que Chinstrap esté "entre" ambas. La salida sigue siendo una categoría discreta sin orden, y el error se mide por acierto o fallo, no por distancia numérica. Si se tratara como regresión, predecir 1,5 no tendría sentido biológico.

### Conclusión

Paradigma, tarea, algoritmo, predicción y decisión son niveles distintos. Un mismo dominio (Palmer Penguins) admite formulaciones supervisadas, no supervisadas y semisupervisadas, y el algoritmo es solo una de las piezas que definen el experimento.

## Ejercicio 2. Tres formulaciones sobre el mismo conjunto de datos

**Objetivo:** distinguir diferentes tareas que pueden formularse sobre un mismo dominio.

### Tabla comparativa

| Elemento | Definición propuesta: clasificación | Definición propuesta: regresión | Definición propuesta: clustering |
|---|---|---|---|
| Uso o decisión que se desea apoyar | Clasificar individuos según su especie (registro, o revisión experta de casos dudosos) | Predecir la masa corporal de un individuo sin pesarlo | Descubrir perfiles morfológicos de pingüinos, sin partir de etiquetas |
| Unidad de análisis | Cada fila representa un pingüino individual | Cada fila representa un pingüino individual | Cada fila representa un pingüino individual |
| Variables de entrada candidatas | `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex` (`island` con cautela: puede ser un atajo) | `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `sex`, `island`, `species` | `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g` (estandarizadas) |
| Variable objetivo, si corresponde | `species` | `body_mass_g` | No corresponde (`species` se reserva para interpretar los grupos después) |
| Salida producida por el modelo | Una etiqueta entre Adelie, Chinstrap y Gentoo, o una probabilidad por especie | Un número continuo y positivo, en gramos | Un número de cluster (0, 1, 2...) para cada pingüino, sin nombre biológico |
| Error o fracaso relevante | Asignar la especie equivocada (por ejemplo, confundir Adelie con Chinstrap) o dar una predicción poco confiable sin revisarla | Error absoluto en gramos (MAE/RMSE); también puede expresarse como error porcentual | Grupos inestables, mal separados o sin interpretación útil (silhouette baja, resultados que cambian con la semilla) |

### Preguntas de análisis

**1. ¿Qué cambia entre las tres formulaciones aunque las filas sean las mismas?**
Cambian el objetivo, la salida, el tipo de error y la forma de evaluar. En clasificación se predice una categoría y se mide el acierto. En regresión se predice un número y se mide cuántos gramos se erra. En clustering no hay objetivo: se buscan grupos y se evalúa su cohesión, su separación y su estabilidad. Los datos son los mismos, pero la pregunta y el criterio de éxito son distintos.

**2. ¿Qué variable debe ocultarse durante el clustering? ¿Para qué podría utilizarse después?**
Debe ocultarse `species` (y también `island` y `sex`). Si entrara en el agrupamiento, los grupos reflejarían la especie por construcción y no habría nada que descubrir. Después se usa como **evaluación externa**: se cruzan los clusters con `species` para ver si los grupos encontrados coinciden con las especies.

**3. En regresión, ¿qué función debe asignarse a `body_mass_g` y qué riesgo produciría incluirla entre los predictores?**
Debe ser la **variable objetivo**. Incluirla como predictor produciría **fuga de información**: el modelo recibiría la respuesta como entrada, y el desempeño sería perfecto pero falso, porque en un caso real la masa es justamente lo que se quiere estimar.

**4. ¿Qué limitaciones de alcance tendrían las conclusiones obtenidas con estos datos?**
Las conclusiones valen solo para la población que representan los datos: tres especies de pingüinos del archipiélago Palmer, en tres islas (Biscoe, Dream y Torgersen) y en los años 2007 a 2009, con 344 observaciones. No se pueden extender a otras especies, otras islas, otros años ni otros procedimientos de medición. Además, `island` y `species` están muy relacionadas, lo que puede dar resultados optimistas si el modelo usa la isla como atajo.

### Comparación experimental

Aunque las tres formulaciones usan las mismas filas, son experimentos distintos. La clasificación requiere un objetivo categórico y se evalúa con métricas de acierto, como accuracy y F1. La regresión requiere un objetivo numérico y se evalúa con errores en gramos, como MAE y RMSE. El clustering no tiene objetivo, por lo que se evalúa con criterios internos (silhouette, estabilidad) y, de forma separada, con un criterio externo (comparar con `species` una vez terminado el agrupamiento). Esto afecta también el papel de cada variable: `species` es objetivo en clasificación, puede ser predictor en regresión y se reserva en clustering; `body_mass_g` es predictor en clasificación y clustering, pero objetivo en regresión.

## Ejercicio 3. Auditoría estructural y reserva temprana de la prueba

**Objetivo:** Definir una separación reproducible antes de explorar relaciones con el objetivo o tomar decisiones de
modelado.

In [1]:
#%pip install pandas numpy matplotlib seaborn scikit-learn

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

RANDOM_STATE = 42
DATA_URL = (
    "https://raw.githubusercontent.com/allisonhorst/"
    "palmerpenguins/master/inst/extdata/penguins.csv"
)

print("pandas:", pd.__version__, "| numpy:", np.__version__, "| scikit-learn:", sklearn.__version__)

pandas: 3.0.5 | numpy: 2.5.2 | scikit-learn: 1.9.0


In [2]:
from datetime import date

datos = pd.read_csv(DATA_URL)
datos_originales = datos.copy(deep=True)
datos_originales.to_csv("penguins_original.csv", index=False)
print("Fecha de descarga:", date.today())

print("Dimensiones:", datos.shape)
print("Columnas:", list(datos.columns))
print(datos.dtypes)
print("species faltantes:", datos["species"].isna().sum())
print("Filas idénticas:", datos.duplicated().sum())

Fecha de descarga: 2026-10-09
Dimensiones: (344, 8)
Columnas: ['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex', 'year']
species                  str
island                   str
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                      str
year                   int64
dtype: object
species faltantes: 0
Filas idénticas: 0


In [3]:
from sklearn.model_selection import train_test_split

idx_desarrollo, idx_prueba = train_test_split(
    datos.index, test_size=0.20,
    stratify=datos["species"], random_state=RANDOM_STATE
)
desarrollo = datos.loc[idx_desarrollo].copy()
prueba = datos.loc[idx_prueba].copy()

pd.Series(idx_desarrollo, name="indice").to_csv("indices_desarrollo.csv", index=False)
pd.Series(idx_prueba, name="indice").to_csv("indices_prueba.csv", index=False)

print("Desarrollo:", len(desarrollo), "| Prueba:", len(prueba))
print("Intersección vacía:", len(set(idx_desarrollo) & set(idx_prueba)) == 0)
print("Unión reconstruye el original:", sorted(set(idx_desarrollo) | set(idx_prueba)) == list(datos.index))

# A partir de acá, no se toca ¨prueba¨ ni ¨desarrollo¨, solo se trabaja con copias de ellos.

Desarrollo: 275 | Prueba: 69
Intersección vacía: True
Unión reconstruye el original: True


In [4]:
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

print("Faltantes:\n", desarrollo.isna().sum())
print("Duplicados exactos:", desarrollo.duplicated().sum())
print("Duplicados lógicos (sin year):",
      desarrollo.duplicated(subset=[c for c in desarrollo.columns if c != "year"]).sum())

print(desarrollo["species"].value_counts())
print(desarrollo["species"].value_counts(normalize=True).round(3))
print(desarrollo["island"].value_counts())
print(desarrollo["sex"].value_counts(dropna=False))

print(desarrollo[medidas].describe().round(2).T)

for c in medidas:   # valores físicamente improbables
    q1, q3 = desarrollo[c].quantile([.25, .75]); iqr = q3 - q1
    fuera = desarrollo[(desarrollo[c] < q1 - 1.5*iqr) | (desarrollo[c] > q3 + 1.5*iqr)]
    print(c, "| <= 0:", (desarrollo[c] <= 0).sum(), "| fuera de 1.5*IQR:", len(fuera))

pd.crosstab(desarrollo["island"], desarrollo["species"], margins=True)

Faltantes:
 species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
dtype: int64
Duplicados exactos: 0
Duplicados lógicos (sin year): 0
species
Adelie       122
Gentoo        99
Chinstrap     54
Name: count, dtype: int64
species
Adelie       0.444
Gentoo       0.360
Chinstrap    0.196
Name: proportion, dtype: float64
island
Biscoe       134
Dream         98
Torgersen     43
Name: count, dtype: int64
sex
male      134
female    130
NaN        11
Name: count, dtype: int64
                   count     mean     std     min     25%     50%     75%  \
bill_length_mm     273.0    44.07    5.50    33.1    39.5    44.9    48.7   
bill_depth_mm      273.0    17.17    2.01    13.1    15.5    17.3    18.8   
flipper_length_mm  273.0   200.85   13.96   172.0   190.0   197.0   213.0   
body_mass_g        273.0  4211.26  805.75  2700.0  3550.0  4050.0  4775.0   

   

species,Adelie,Chinstrap,Gentoo,All
island,,,,
Biscoe,35,0,99,134
Dream,44,54,0,98
Torgersen,43,0,0,43
All,122,54,99,275


In [5]:
def resumen(col):
    s = desarrollo[col].dropna()
    if pd.api.types.is_numeric_dtype(s):
        return f"{s.min():g} – {s.max():g}"
    return ", ".join(f"{k} ({v})" for k, v in s.value_counts().items())

diccionario_auto = pd.DataFrame({
    "Tipo en pandas": desarrollo.dtypes.astype(str),
    "Faltantes": desarrollo.isna().sum(),
    "Valores o rango": [resumen(c) for c in desarrollo.columns],
})
diccionario_auto

,Tipo en pandas,Faltantes,Valores o rango
species,str,0,"Adelie (122), Gentoo (99), Chinstrap (54)"
island,str,0,"Biscoe (134), Dream (98), Torgersen (43)"
bill_length_mm,float64,2,33.1 – 59.6
bill_depth_mm,float64,2,13.1 – 21.5
flipper_length_mm,float64,2,172 – 231
body_mass_g,float64,2,2700 – 6300
sex,str,11,"male (134), female (130)"
year,int64,0,2007 – 2009


### Protocolo de partición

- **Unidad de análisis:** pingüino individual. **Caso nuevo:** otro individuo de la misma población de referencia (islas Biscoe, Dream y Torgersen; años 2007 a 2009).
- **Criterio:** partición aleatoria estratificada por `species`, 80 % desarrollo y 20 % prueba.
- **Semilla:** `RANDOM_STATE = 42`. Índices guardados en `indices_desarrollo.csv` e `indices_prueba.csv`.
- **Verificaciones:** el archivo completo tiene 344 filas y 8 columnas, `species` no tiene faltantes y hay 0 filas idénticas. Desarrollo tiene 275 filas y prueba 69; la intersección es vacía y la unión reconstruye el original.

### Diccionario de datos

| Variable | Descripción | Tipo conceptual | Tipo en pandas | Unidad o categorías | Faltantes | Valores o rango observados | Rol posible | Tratamiento, controles y riesgos |
|---|---|---|---|---|---|---|---|---|
| `species` | Especie del pingüino | Categórica nominal | object / str | Adelie, Chinstrap, Gentoo | 0 | Adelie (122), Gentoo (99), Chinstrap (54) | Objetivo en clasificación; reservada en clustering; predictor opcional en regresión | Codificar solo si el algoritmo lo requiere. Clases desbalanceadas: estratificar. Riesgo de fuga si entra como predictor en la clasificación |
| `island` | Isla de observación | Categórica nominal | object / str | Biscoe, Dream, Torgersen | 0 | Biscoe (134), Dream (98), Torgersen (43) | Variable contextual; predictor con cautela | Codificación one-hot dentro del pipeline. Riesgo de atajo: casi determina la especie |
| `bill_length_mm` | Longitud del pico | Numérica continua | float64 | mm | 2 | 33,1 – 59,6 | Predictor | Imputar con mediana dentro del pipeline; escalar si el algoritmo lo requiere |
| `bill_depth_mm` | Profundidad del pico | Numérica continua | float64 | mm | 2 | 13,1 – 21,5 | Predictor | Igual que la anterior |
| `flipper_length_mm` | Longitud de la aleta | Numérica continua | float64 | mm | 2 | 172 – 231 | Predictor | Igual que la anterior |
| `body_mass_g` | Masa corporal | Numérica continua | float64 | g | 2 | 2700 – 6300 | Predictor en clasificación y clustering; objetivo en regresión | Imputar dentro del pipeline. Nunca incluirla como predictor si es el objetivo (fuga) |
| `sex` | Sexo registrado | Categórica nominal | object / str | female, male | 11 | female (130), male (134) | Predictor opcional | Imputar con la moda dentro del pipeline y codificar. Es la variable con más faltantes |
| `year` | Año de observación | Numérica discreta (ordinal) | int64 | 2007, 2008, 2009 | 0 | 2007 (90), 2008 (84), 2009 (101) | Variable contextual; usada en la extensión de transferencia | No usar como número continuo. No incluir como predictor en la tarea principal |

### Tabla de calidad (desarrollo)

| Control | Resultado |
|---|---|
| Faltantes en medidas corporales | 2 por variable (las mismas 2 filas) |
| Faltantes en `sex` | 11 |
| Faltantes en `species`, `island`, `year` | 0 |
| Filas exactamente duplicadas | 0 |
| Duplicados lógicos (todas las columnas salvo `year`) | 0 |
| Valores menores o iguales a 0 en las medidas | 0 |
| Valores fuera de 1,5·IQR | 0 |

**Valores físicamente improbables:** no se encontraron. Los rangos son coherentes con las unidades (mm y g), sin señales de mezcla de unidades. Si aparecieran, se conservarían durante la auditoría, se verificarían contra la fuente y, ante un error evidente de medición, se marcarían como faltantes para que los trate la imputación dentro del pipeline, sin eliminarlos en silencio.

### Tabla cruzada `island` por `species` (desarrollo)

| island | Adelie | Chinstrap | Gentoo | Total |
|---|---|---|---|---|
| Biscoe | 35 | 0 | 99 | 134 |
| Dream | 44 | 54 | 0 | 98 |
| Torgersen | 43 | 0 | 0 | 43 |
| Total | 122 | 54 | 99 | 275 |

### Preguntas de análisis

**1. ¿Qué tratamiento requieren los faltantes antes de entrenar?**
Imputación aprendida solo con los datos de entrenamiento de cada iteración, dentro de un `Pipeline`: mediana para las medidas y moda para `sex`. No se imputa antes de la partición ni con estadísticas de la prueba. Las 2 filas sin ninguna medida no aportan información morfológica: se conservan para mantener el protocolo único y se documenta que su imputación es puramente "promedio"; la alternativa sería excluirlas del entrenamiento.

**2. ¿En qué datos debe calcularse la media dentro de cada fold y qué fuga produciría calcularla sobre desarrollo completo?**
Debe calcularse solo con las filas de entrenamiento de ese fold. Si se calcula sobre todo desarrollo, la media incorpora valores del fold de validación, de modo que ese fold deja de ser independiente y la estimación resulta algo optimista.

**3. ¿`island` podría actuar como atajo para predecir `species`?**
Sí. En la tabla cruzada, Torgersen contiene solo Adelie (43 de 43); Dream contiene Adelie y Chinstrap, y Biscoe contiene Adelie y Gentoo. Una regla que asigne la especie más frecuente de cada isla ya acierta 196 de 275 casos (aprox. 71 %) sin mirar ninguna medida corporal. Por eso `island` se trata con cautela y su aporte debe compararse con y sin ella.

**4. ¿El año debería tratarse como número continuo o categoría?**
Como categoría u ordinal discreta. Solo hay tres valores y no se busca extrapolar una tendencia lineal a otros años: tratarlo como continuo impondría ese supuesto. Para la tarea principal (casos nuevos del mismo contexto) no se incluye como predictor; se usa en la extensión de transferencia temporal.

**5. ¿Qué información fue admisible para elegir la partición y qué análisis debió esperar hasta reservar la prueba?**
Fue admisible la auditoría estructural: dimensiones, nombres, tipos, disponibilidad del objetivo, identificadores, duplicados y `species` para estratificar. Debieron esperar las asociaciones con el objetivo (como la tabla `island` por `species`), los rankings de variables, las estadísticas de las medidas, las imputaciones, las escalas y cualquier resultado de modelos.

### Variables candidatas y variables con cautela

- **Candidatas:** `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g` (con imputación dentro del pipeline).
- **Con cautela:** `island` (atajo para `species`), `sex` (11 faltantes) y `year` (variable contextual, no predictor en la tarea principal).
- **Objetivo:** `species`.

### Conclusión

La prueba (69 filas) queda reservada con semilla 42 e índices guardados, y no se consulta hasta el Ejercicio 13. El conjunto de desarrollo (275 filas) no tiene duplicados ni valores improbables; sus problemas de calidad son los faltantes, que se tratan dentro del pipeline, y el riesgo de atajo de `island`.

## Ejercicio 4. Papeles de desarrollo, validación y prueba

**Objetivo:** asignar a cada partición un único papel experimental.

In [6]:
import json
from sklearn.model_selection import StratifiedKFold

idx_d = pd.read_csv("indices_desarrollo.csv")["indice"].to_numpy()
idx_p = pd.read_csv("indices_prueba.csv")["indice"].to_numpy()
desarrollo_rec = datos_originales.loc[idx_d]
prueba_rec = datos_originales.loc[idx_p]
n = len(datos_originales)

print(f"Desarrollo: {len(desarrollo_rec)} ({len(desarrollo_rec)/n:.1%}) | Prueba: {len(prueba_rec)} ({len(prueba_rec)/n:.1%})")
print("Coincide con la partición del Ejercicio 3:",
      set(idx_d) == set(idx_desarrollo) and set(idx_p) == set(idx_prueba))

Desarrollo: 275 (79.9%) | Prueba: 69 (20.1%)
Coincide con la partición del Ejercicio 3: True


In [7]:
cantidades = pd.DataFrame({
    "Completo": datos_originales["species"].value_counts(),
    "Desarrollo": desarrollo_rec["species"].value_counts(),
    "Prueba": prueba_rec["species"].value_counts(),
})
proporciones = (cantidades / cantidades.sum()).round(3)
tabla_estratificacion = pd.concat([cantidades, proporciones.add_suffix(" (prop.)")], axis=1)
tabla_estratificacion

,Completo,Desarrollo,Prueba,Completo (prop.),Desarrollo (prop.),Prueba (prop.)
species,,,,,,
Adelie,152,122,30,0.442,0.444,0.435
Gentoo,124,99,25,0.360,0.360,0.362
Chinstrap,68,54,14,0.198,0.196,0.203


In [8]:
K_FOLDS = 5
cv = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_STATE)

y_dev = desarrollo_rec["species"].reset_index(drop=True)
filas = []
for i, (tr, va) in enumerate(cv.split(np.zeros(len(y_dev)), y_dev), start=1):
    filas.append({"fold": i, "entrenamiento": len(tr), "validación": len(va),
                  **y_dev.iloc[va].value_counts().to_dict()})
pd.DataFrame(filas)

,fold,entrenamiento,validación,Adelie,Gentoo,Chinstrap
0,1,220,55,25,20,10
1,2,220,55,25,19,11
2,3,220,55,24,20,11
3,4,220,55,24,20,11
4,5,220,55,24,20,11


In [9]:
manifiesto = {
    "criterio": "aleatoria estratificada por species",
    "test_size": 0.20,
    "random_state": RANDOM_STATE,
    "n_total": int(n),
    "n_desarrollo": int(len(desarrollo_rec)),
    "n_prueba": int(len(prueba_rec)),
    "conteos_especie": {k: {e: int(v) for e, v in cantidades[k].items()} for k in cantidades},
    "proporciones_especie": {k: {e: float(v) for e, v in proporciones[k].items()} for k in proporciones},
    "validacion": {"metodo": "StratifiedKFold", "n_splits": K_FOLDS, "shuffle": True, "random_state": RANDOM_STATE},
    "archivos_indices": ["indices_desarrollo.csv", "indices_prueba.csv"],
}
with open("manifiesto_particion.json", "w", encoding="utf-8") as f:
    json.dump(manifiesto, f, indent=2, ensure_ascii=False)
print(json.dumps(manifiesto, indent=2, ensure_ascii=False))

{
  "criterio": "aleatoria estratificada por species",
  "test_size": 0.2,
  "random_state": 42,
  "n_total": 344,
  "n_desarrollo": 275,
  "n_prueba": 69,
  "conteos_especie": {
    "Completo": {
      "Adelie": 152,
      "Gentoo": 124,
      "Chinstrap": 68
    },
    "Desarrollo": {
      "Adelie": 122,
      "Gentoo": 99,
      "Chinstrap": 54
    },
    "Prueba": {
      "Adelie": 30,
      "Gentoo": 25,
      "Chinstrap": 14
    }
  },
  "proporciones_especie": {
    "Completo": {
      "Adelie": 0.442,
      "Gentoo": 0.36,
      "Chinstrap": 0.198
    },
    "Desarrollo": {
      "Adelie": 0.444,
      "Gentoo": 0.36,
      "Chinstrap": 0.196
    },
    "Prueba": {
      "Adelie": 0.435,
      "Gentoo": 0.362,
      "Chinstrap": 0.203
    }
  },
  "validacion": {
    "metodo": "StratifiedKFold",
    "n_splits": 5,
    "shuffle": true,
    "random_state": 42
  },
  "archivos_indices": [
    "indices_desarrollo.csv",
    "indices_prueba.csv"
  ]
}


### Partición utilizada

Se utiliza exactamente la separación fijada en el Ejercicio 3, reconstruida a partir de los índices guardados: desarrollo con 275 filas (79,9 %) y prueba con 69 filas (20,1 %). La reconstrucción coincide con la partición original.

### Validación dentro de desarrollo

Validación cruzada estratificada por `species`, con K = 5, mezcla de filas y `random_state = 42`. En cada fold, 220 filas entrenan y 55 validan, con 10 o 11 Chinstrap en cada validación. Con la validación se podrán tomar decisiones sobre variables, preprocesamiento, familia de modelo e hiperparámetros. La prueba no interviene en ninguna de esas decisiones.

### Proporciones de especies

| Especie | Completo | Desarrollo | Prueba | Prop. completo | Prop. desarrollo | Prop. prueba |
|---|---|---|---|---|---|---|
| Adelie | 152 | 122 | 30 | 0,442 | 0,444 | 0,435 |
| Gentoo | 124 | 99 | 25 | 0,360 | 0,360 | 0,362 |
| Chinstrap | 68 | 54 | 14 | 0,198 | 0,196 | 0,203 |
| Total | 344 | 275 | 69 | 1,000 | 1,000 | 1,000 |

La estratificación mantuvo las proporciones: la diferencia máxima respecto al conjunto completo es de 0,002 en desarrollo y de 0,007 en prueba.

### Manifiesto de partición

| Elemento | Valor |
|---|---|
| Criterio | Aleatoria estratificada por `species` |
| Proporción de prueba | 0,20 |
| Semilla | 42 |
| Índices | `indices_desarrollo.csv` e `indices_prueba.csv` |
| Cantidades | 344 en total, 275 en desarrollo y 69 en prueba |
| Validación | `StratifiedKFold`, K = 5, `shuffle=True`, semilla 42 |
| Archivo del manifiesto | `manifiesto_particion.json` |

### Funciones y momentos de uso

| Partición | Función asignada | Momento de uso |
|---|---|---|
| Entrenamiento | Ajustar los parámetros del modelo y aprender imputación, codificación y escalado | Dentro de cada fold (220 filas) y, al final, con todo desarrollo (275 filas) para el reajuste |
| Validación | Comparar modelos, hiperparámetros y variables, y estimar desempeño y variabilidad | Durante el desarrollo (Ejercicios 6 a 12), sobre las 55 filas reservadas de cada fold |
| Prueba | Estimación final independiente del desempeño | Una sola vez, en el Ejercicio 13, con la configuración congelada |

### Preguntas de análisis

**1. ¿Por qué la estratificación es apropiada para este objetivo?**
Porque `species` está desbalanceada (Chinstrap es el 19,8 %) y la prueba es chica (69 filas, 14 Chinstrap). Con una partición aleatoria simple, el azar podría dejar muy pocos Chinstrap en un conjunto y distorsionar las métricas por especie. Además, el escenario de uso son individuos nuevos de la misma población, así que conviene que las proporciones se conserven.

**2. ¿Qué ocurriría si una clase poco frecuente quedara casi ausente de validación?**
Las métricas de esa clase (recall, F1) se calcularían con uno o dos casos y serían inestables. La accuracy global casi no lo notaría, y se podría elegir un modelo que falla justo en esa especie sin darse cuenta.

**3. ¿Por qué consultar la prueba después de cada modificación invalida su papel?**
Porque la prueba pasa a funcionar como validación: cada elección se ajusta, aunque sea sin querer, a sus particularidades. Con muchas consultas, el resultado final es optimista y deja de ser una estimación sobre datos independientes.

**4. ¿Los porcentajes elegidos constituyen una regla universal? ¿Qué otros factores deberían considerarse?**
No. Dependen del tamaño del conjunto, de los casos de la clase minoritaria, de la variabilidad que se tolera en la estimación, del escenario de uso, del costo de los errores, de la cantidad de modelos a comparar y de si existen dependencias entre filas (grupos o tiempo). Con 14 Chinstrap en prueba, por ejemplo, la estimación por especie ya es bastante ruidosa.

**5. Si se descubre una dependencia entre filas que vuelve incorrecta la separación, ¿por qué debe redefinirse el experimento antes de seguir comparando modelos?**
Porque las filas dependientes (por ejemplo, el mismo individuo) quedarían a ambos lados de la partición y los resultados serían optimistas. Toda comparación hecha sobre esa base sería inválida. Hay que redefinir la unidad de análisis y el criterio de partición (por ejemplo, por grupos) y reservar una nueva prueba.

### Conclusión

La partición es reproducible, está estratificada y asigna un papel único a cada conjunto: el desarrollo se usa para entrenar y decidir mediante validación cruzada, y la prueba permanece cerrada hasta el Ejercicio 13.

## Ejercicio 5. Detectives de fuga de información

**Objetivo:** identificar procedimientos que producen evaluaciones demasiado optimistas.

**Criterio de diagnóstico.** Un procedimiento está contaminado si (a) algo aprendido de los datos (media, escala, ranking de variables, elección de configuración) usa filas que después sirven para evaluar, o (b) usa información que no existiría en el momento real de predecir un caso nuevo.

### Diagnóstico de los diez casos

| # | Situación | Diagnóstico | Explicación: qué información cruza la frontera y cuándo se genera | Corrección |
|---|---|---|---|---|
| 1 | Imputar con la media de todas las filas y luego separar | Contaminada | La media incluye valores de las filas que luego serán de prueba. En la predicción real, los casos nuevos no existen todavía cuando se calcula esa media | Separar primero y ajustar el imputador solo con entrenamiento, dentro de un `Pipeline` |
| 2 | Separar primero y ajustar el imputador con entrenamiento | Correcta | Las estadísticas se aprenden solo con entrenamiento y se aplican a la prueba, igual que ocurriría con un caso nuevo | No requiere corrección |
| 3 | Estandarizar las cuatro variables con todos los datos antes de la validación cruzada | Contaminada | La media y el desvío incluyen las filas de validación de cada fold, así que ese fold ya no es independiente | Incluir `StandardScaler` dentro del `Pipeline`, de modo que se ajuste con el entrenamiento de cada fold |
| 4 | Elegir las dos variables más correlacionadas con el objetivo usando entrenamiento y prueba juntos | Contaminada | La selección de variables usa las etiquetas de la prueba. Esa información (la especie de los casos de prueba) no se conoce al predecir | Hacer la selección solo con desarrollo y, dentro de la validación cruzada, solo con el entrenamiento de cada fold |
| 5 | Probar veinte configuraciones y quedarse con la de mayor accuracy en prueba | Contaminada | La prueba se usa para decidir. Entre veinte configuraciones, la mejor en prueba lo es en parte por azar, y su accuracy resulta optimista | Elegir con validación cruzada en desarrollo, congelar la configuración y consultar la prueba una sola vez |
| 6 | Mantener las fotografías del mismo pingüino en una única partición | Correcta | Evita que un mismo individuo aparezca en entrenamiento y en prueba. Equivale a particionar por grupos | No requiere corrección. En Palmer Penguins cada fila ya es un individuo distinto |
| 7 | Incluir una variable creada después de confirmar la especie | Contaminada | La variable se genera usando la respuesta (fuga por variable). En la predicción real la especie aún no se conoce, así que esa variable no existiría | Excluirla y usar solo variables disponibles en el momento de predecir |
| 8 | Eliminar duplicados después de repartir sus copias entre entrenamiento y prueba | Contaminada | Una copia está en entrenamiento y la otra en prueba: el modelo ya vio esa observación. Eliminar duplicados después no repara la contaminación ya producida | Deduplicar antes de particionar (o particionar por grupos). En esta versión del conjunto hay 0 filas idénticas |
| 9 | Ajustar el escalador dentro de cada fold de validación cruzada | Correcta | En cada fold, el escalador aprende solo del entrenamiento y se aplica al fold reservado | No requiere corrección |
| 10 | Usar `island` sin analizar que puede actuar como atajo para `species` | Contaminada (atajo no auditado) | No es una fuga temporal: `island` sí se conoce al predecir. El problema es que el modelo puede apoyarse en una regla de contexto y no en la morfología: en desarrollo, asignar la especie más frecuente de cada isla ya acierta aprox. 71 % sin usar ninguna medida. Esa relación depende de cómo se muestreó y puede no sostenerse en un contexto nuevo | Analizar la tabla cruzada, comparar el desempeño con y sin `island`, documentar la decisión y evaluar la transferencia por isla en la extensión |

### Resumen por tipo de problema

- **Transformaciones aprendidas fuera del ciclo** (casos 1 y 3): imputación y escalado.
- **Selección de variables** (caso 4).
- **Reutilización de la prueba** (caso 5).
- **Entidades y duplicados** (casos 6 y 8).
- **Variables que no existirían al predecir** (caso 7).
- **Atajos de contexto** (caso 10).

### Conclusión

Los casos correctos (2, 6 y 9) comparten un patrón: todo lo que aprende de los datos se aprende solo con entrenamiento, y la evaluación se hace sobre filas que el procedimiento no vio. En los contaminados, la evaluación resulta optimista porque la métrica mide algo distinto a lo que ocurrirá con casos nuevos.

## Ejercicio 6. Imputación y escalado: procedimiento incorrecto y procedimiento válido

**Objetivo:** demostrar que las transformaciones también aprenden de los datos.

In [10]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

predictores = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X_des = desarrollo_rec[predictores]
y_des = desarrollo_rec["species"]

# Mismos folds que en el Ejercicio 4
cv = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_STATE)

def nuevo_modelo():
    return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

metricas = {"accuracy": "accuracy", "f1_macro": "f1_macro", "neg_log_loss": "neg_log_loss"}

In [11]:
imp_A = SimpleImputer(strategy="mean")
esc_A = StandardScaler()
X_A = esc_A.fit_transform(imp_A.fit_transform(X_des))   # se ajusta con TODO desarrollo
res_A = cross_validate(nuevo_modelo(), X_A, y_des, cv=cv, scoring=metricas)

In [12]:
pipeline = Pipeline([
    ("imputador", SimpleImputer(strategy="mean")),
    ("escalador", StandardScaler()),
    ("modelo", nuevo_modelo()),
])
res_B = cross_validate(pipeline, X_des, y_des, cv=cv, scoring=metricas)   # se reajusta en cada fold

In [13]:
def fila(res, nombre):
    return {
        "Diseño": nombre,
        "accuracy": res["test_accuracy"].mean(),
        "accuracy (desvío)": res["test_accuracy"].std(),
        "F1 macro": res["test_f1_macro"].mean(),
        "F1 macro (desvío)": res["test_f1_macro"].std(),
        "log-loss": -res["test_neg_log_loss"].mean(),
    }

comparacion = pd.DataFrame([fila(res_A, "A (contaminado)"), fila(res_B, "B (pipeline)")]).round(4)
display(comparacion)
print("Accuracy por fold, A:", res_A["test_accuracy"].round(4))
print("Accuracy por fold, B:", res_B["test_accuracy"].round(4))

,Diseño,accuracy,accuracy (desvío),F1 macro,F1 macro (desvío),log-loss
0,A (contaminado),0.9782,0.0291,0.9733,0.0351,0.0688
1,B (pipeline),0.9782,0.0291,0.9733,0.0351,0.0686


Accuracy por fold, A: [0.9636 1.     1.     0.9273 1.    ]
Accuracy por fold, B: [0.9636 1.     1.     0.9273 1.    ]


In [14]:
print("Con todo desarrollo (Diseño A):")
print("  medias del imputador:", imp_A.statistics_.round(2))
print("  desvíos del escalador:", esc_A.scale_.round(2))

filas = []
for i, (tr, va) in enumerate(cv.split(X_des, y_des), start=1):
    p = Pipeline([("imputador", SimpleImputer(strategy="mean")),
                  ("escalador", StandardScaler())]).fit(X_des.iloc[tr])
    filas.append({"fold": i,
                  "media body_mass_g": p["imputador"].statistics_[3],
                  "desvío body_mass_g": p["escalador"].scale_[3]})
pd.DataFrame(filas).round(2)

Con todo desarrollo (Diseño A):
  medias del imputador: [  44.07   17.17  200.85 4211.26]
  desvíos del escalador: [  5.47   2.    13.89 801.34]


,fold,media body_mass_g,desvío body_mass_g
0,1,4219.15,801.45
1,2,4215.14,793.59
2,3,4190.83,804.21
3,4,4216.55,802.40
4,5,4214.61,804.67


**Datos y configuración.** Predictores: `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm` y `body_mass_g`. Objetivo: `species`. Se trabaja solo con el conjunto de desarrollo. Ambos diseños usan los mismos 5 folds estratificados (semilla 42) y la misma `LogisticRegression` (`max_iter=1000`). Imputación con la media.

### Pasos de ambos diseños

**Diseño A (contaminado, solo diagnóstico):**
1. Calcular las medias con todo desarrollo e imputar.
2. Calcular media y desvío con todo desarrollo y estandarizar.
3. Aplicar la validación cruzada sobre los datos ya transformados.

**Diseño B (válido):**
1. Construir un `Pipeline` con imputador, escalador y regresión logística.
2. En cada fold, ajustar imputador y escalador solo con las filas de entrenamiento del fold.
3. Aplicar exactamente esos parámetros aprendidos al fold reservado y evaluar.

### Resultados

| Diseño | accuracy | desvío | F1 macro | desvío | log-loss |
|---|---|---|---|---|---|
| A (contaminado) | 0,9782 | 0,0291 | 0,9733 | 0,0351 | 0,0688 |
| B (pipeline) | 0,9782 | 0,0291 | 0,9733 | 0,0351 | 0,0686 |

La accuracy y el F1 macro son iguales en los cinco folds. El log-loss difiere solo en el cuarto decimal, con signo cambiante entre folds.

Ejemplo de lo que aprende el preprocesamiento: la media de `body_mass_g` es 4211,26 g con todo desarrollo, y entre 4190,83 y 4219,15 g según el fold de entrenamiento.

### Preguntas de análisis

**1. ¿Por qué el Diseño A está conceptualmente invalidado aunque produzca una métrica similar?**
Porque la validez de un procedimiento depende de qué información usa, no de si la métrica cambia. En A, la media y el desvío incluyen las filas de validación, de modo que cada fold reservado influyó en el preprocesamiento. Que aquí la diferencia sea nula se debe a que el conjunto tiene 275 filas, pocos faltantes y un modelo poco sensible. Con otros datos o transformaciones (por ejemplo, selección de variables) la diferencia puede ser grande, y no se puede saber de antemano.

**2. ¿Qué parámetros aprende el imputador?**
La media de cada columna, calculada ignorando los valores faltantes: cuatro valores, uno por predictor.

**3. ¿Qué parámetros aprende el escalador?**
La media y el desvío estándar de cada columna: ocho valores en total.

**4. ¿Qué objeto se evalúa realmente: el algoritmo final o el procedimiento completo?**
El procedimiento completo: imputación, escalado y modelo juntos. La validación cruzada sobre el pipeline reajusta todo en cada fold.

### Conclusión

El Diseño B es el único que respeta la independencia de la evaluación. Ambos diseños dieron resultados prácticamente iguales, lo que muestra que una métrica parecida no prueba que el procedimiento sea válido.